# DDSM115 4WD-A: MJX/brax PPO training

Train two RL tasks for the Waveshare DDSM115 4WD-A skid-steer robot in MJX (the JAX port of MuJoCo) using brax PPO.

1. **DriveToGoal** — navigate to a random xy goal
2. **VelocityTracking** — track a commanded `(vx, omega_z)`

Both share a base env defined in `rl/base.py`; this notebook trains each with the same PPO hyper-parameters and rolls out the trained policy on CPU for inspection.

**Runtime:** enable a GPU runtime in Colab (`Runtime` → `Change runtime type` → `T4 GPU` or better). On a T4, ~10M env steps trains DriveToGoal to a reasonable policy in ~5 min.

## 1. Install dependencies

Colab ships with a recent JAX; brax 0.10.5 wants jax 0.4.x. We force-install the pinned versions and then **restart the runtime** automatically. After the runtime restart, re-run from the next cell.

If you skip the restart, `import brax` may still work but the JAX it imports against will be the old preinstalled one, and you'll get cryptic shape / pytree errors mid-training.

In [ ]:
# Install pinned versions. Don't use --quiet: surface failures.
!pip install --upgrade --force-reinstall \
    "jax==0.4.35" "jaxlib==0.4.35" \
    "mujoco==3.2.7" "mujoco-mjx==3.2.7" \
    "brax==0.10.5" "flax==0.9.0" \
    "optax==0.2.4" "orbax-checkpoint==0.6.4" \
    mediapy

In [ ]:
# Restart the kernel so the freshly-installed JAX is the one imported.
# After this cell auto-restarts the kernel, RUN THE NEXT CELL.
import os
os.kill(os.getpid(), 9)

## 2. Clone the repo and verify versions

In [ ]:
import os, subprocess, sys
REPO = 'https://github.com/onetxpanda/4wd-mujoco.git'
BRANCH = 'claude/mujoco-ddsm115-robot-cklql'

# Idempotent clone: if we're already inside the repo, do nothing.
if not os.path.exists('ddsm115_4wd.xml'):
    if not os.path.isdir('4wd-mujoco'):
        subprocess.run(['git', 'clone', '-b', BRANCH, REPO, '4wd-mujoco'], check=True)
    os.chdir('4wd-mujoco')

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print('cwd:', os.getcwd())
print('mjcf exists:', os.path.exists('ddsm115_4wd.xml'))

In [ ]:
# Verify every required import succeeds, with versions, before training.
import jax, jaxlib, mujoco, brax, flax, optax
from mujoco import mjx
from brax.envs.base import Env, State
from brax.training.agents.ppo import train as ppo
print('jax     :', jax.__version__)
print('jaxlib  :', jaxlib.__version__)
print('mujoco  :', mujoco.__version__)
print('mjx     : OK')
print('brax    :', brax.__version__)
print('flax    :', flax.__version__)
print('optax   :', optax.__version__)
print('devices :', jax.devices())

## 3. Verify the envs compile + step

In [ ]:
import jax.numpy as jp
from rl import DriveToGoal, VelocityTracking

for cls in (DriveToGoal, VelocityTracking):
    env = cls()
    reset = jax.jit(env.reset)
    step = jax.jit(env.step)
    state = reset(jax.random.PRNGKey(0))
    state = step(state, jp.ones(env.action_size))
    print(f'{cls.__name__}: obs={env.observation_size}d  act={env.action_size}d  '
          f'reward={float(state.reward):+.3f}')

## 4. Train DriveToGoal with PPO

Knobs you'll likely tune:
* `num_timesteps` — total env steps. 10M is a reasonable starter; bump to 50M for a polished policy.
* `num_envs` — parallel rollouts. 2048 fits in T4 VRAM for this model.
* `episode_length` — env step horizon. 500 steps × 10-frame substep × 0.002s = 10 s.

In [ ]:
import time
from rl import DriveToGoal

history = []
t0 = time.time()
def progress(steps, metrics):
    r = float(metrics.get('eval/episode_reward', 0.0))
    history.append((steps, r, time.time() - t0))
    print(f'step={steps:>8d}  reward={r:+.3f}  elapsed={time.time()-t0:.1f}s')

make_policy, params, _ = ppo.train(
    environment=DriveToGoal(),
    num_timesteps=10_000_000,
    num_evals=20,
    reward_scaling=1.0,
    episode_length=500,
    normalize_observations=True,
    action_repeat=1,
    unroll_length=32,
    num_minibatches=16,
    num_updates_per_batch=4,
    discounting=0.99,
    learning_rate=3e-4,
    entropy_cost=1e-3,
    num_envs=2048,
    batch_size=256,
    seed=0,
    progress_fn=progress,
)
drive_to_goal_params = params
drive_to_goal_make_policy = make_policy

In [ ]:
import matplotlib.pyplot as plt
steps, rewards, _ = zip(*history)
plt.plot(steps, rewards); plt.xlabel('env steps'); plt.ylabel('eval reward')
plt.title('DriveToGoal'); plt.grid(True); plt.show()

## 5. Train VelocityTracking with PPO

In [ ]:
from rl import VelocityTracking

history2 = []
t0 = time.time()
def progress2(steps, metrics):
    r = float(metrics.get('eval/episode_reward', 0.0))
    history2.append((steps, r, time.time() - t0))
    print(f'step={steps:>8d}  reward={r:+.3f}  elapsed={time.time()-t0:.1f}s')

make_policy_v, params_v, _ = ppo.train(
    environment=VelocityTracking(),
    num_timesteps=10_000_000,
    num_evals=20,
    reward_scaling=1.0,
    episode_length=500,
    normalize_observations=True,
    action_repeat=1,
    unroll_length=32,
    num_minibatches=16,
    num_updates_per_batch=4,
    discounting=0.99,
    learning_rate=3e-4,
    entropy_cost=1e-3,
    num_envs=2048,
    batch_size=256,
    seed=0,
    progress_fn=progress2,
)
vel_track_params = params_v
vel_track_make_policy = make_policy_v

steps, rewards, _ = zip(*history2)
plt.plot(steps, rewards); plt.xlabel('env steps'); plt.ylabel('eval reward')
plt.title('VelocityTracking'); plt.grid(True); plt.show()

## 6. Roll out and render (offscreen)

Replays the trained DriveToGoal policy in CPU MuJoCo and renders a clip. Swap the env class and params for VelocityTracking.

In [ ]:
import numpy as np, mediapy, mujoco
from rl import DriveToGoal
env = DriveToGoal()
policy = drive_to_goal_make_policy(drive_to_goal_params, deterministic=True)
jit_reset = jax.jit(env.reset)
jit_step  = jax.jit(env.step)
rng = jax.random.PRNGKey(7)
state = jit_reset(rng)

mj_model = env._mj_model
mj_data = mujoco.MjData(mj_model)
renderer = mujoco.Renderer(mj_model, height=320, width=480)
cam = mujoco.MjvCamera()
cam.azimuth = 90; cam.elevation = -55; cam.distance = 4.0
frames = []
for _ in range(400):
    rng, sub = jax.random.split(rng)
    action, _ = policy(state.obs, sub)
    state = jit_step(state, action)
    mj_data.qpos[:] = np.asarray(state.pipeline_state.qpos)
    mj_data.qvel[:] = np.asarray(state.pipeline_state.qvel)
    mujoco.mj_forward(mj_model, mj_data)
    renderer.update_scene(mj_data, camera=cam)
    frames.append(renderer.render())
    if bool(state.done):
        break
mediapy.show_video(frames, fps=50)